# Inheritance, MRO, and Dunder Methods
Instead of treating **Inheritance, MRO, and Dunder Methods** as a small isolated topic, the goal here is to treat it as part of Python's larger runtime story: objects are created, names are bound, frames execute bytecode, and the interpreter keeps following rules whether or not the source code looks simple from the outside.

If this topic ever felt a little too neat or too magical when explained quickly, that is exactly what this rewrite is trying to fix. We are going to keep asking two grounding questions: what changed in memory, and what instructions is the interpreter actually stepping through under the surface?

The point is not to drown in internals. The point is to make the language feel explainable. Once that happens, even advanced behavior starts feeling much less mysterious.


When people struggle with **Inheritance, MRO, and Dunder Methods**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- Understand single inheritance cleanly
- See MRO as a lookup path
- Use `super()` with more confidence
- Treat dunder methods as protocol hooks


The important runtime fact is that methods usually live on classes, not inside each instance. Instances rely on class lookup, and inheritance broadens the path Python searches through when resolving names.


In [1]:
class A:
    value = "A"
class B(A):
    pass
obj = B()
print(obj.__class__)
print(B.__mro__)
print(obj.value)


<class '__main__.B'>
(<class '__main__.B'>, <class '__main__.A'>, <class 'object'>)
A


Disassembly of a method call on a custom object is a reminder that Python is loading attributes and then calling what it finds. Dunder methods are the protocol entry points that make this all work smoothly with built-in syntax.


In [2]:
import dis

class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

def add_two(a, b):
    return a + b

dis.dis(add_two)


 10           0 RESUME                   0

 11           2 LOAD_FAST                0 (a)
              4 LOAD_FAST                1 (b)
              6 BINARY_OP                0 (+)
             10 RETURN_VALUE


The child class can use parent behavior if it does not override it.

It tells Python where to look and in what order.

It is more subtle and more useful than “call the parent directly”.

They plug custom objects into Python syntax and built-ins.


The subclass changes one behavior but still fits the parent idea.


In [3]:
class Animal:
    def speak(self):
        return "..."

class Dog(Animal):
    def speak(self):
        return "woof"

print(Dog().speak())


woof


The subclass extends the parent setup instead of replacing it entirely.


In [4]:
class Person:
    def __init__(self, name):
        self.name = name

class Employee(Person):
    def __init__(self, name, role):
        super().__init__(name)
        self.role = role

e = Employee("Ada", "Engineer")
print(e.name, e.role)


Ada Engineer


These dunder methods let the object behave more naturally.


In [5]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"
    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

print(Vector(1, 2) + Vector(3, 4))


Vector(4, 6)


This is not only a classroom topic. **Inheritance, MRO, and Dunder Methods** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Using inheritance only to avoid typing repeated code
- Forgetting that `super()` follows MRO rather than a simplistic “parent only” rule
- Adding dunder methods without making semantics intuitive


- What is MRO?
- What does `super()` do?
- Why are dunder methods important?


- Inheritance is lookup through class relationships.
- MRO is the search order.
- Dunder methods are protocol hooks.


If this notebook did its job, **Inheritance, MRO, and Dunder Methods** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.


A better way to understand Inheritance, MRO, and Dunder Methods is to keep moving back and forth between three views of the same code.

The first view is the human view. This is what the code is trying to say. The second view is the object view. In that view, we ask which Python objects exist, which names refer to them, and which objects are being created, reused, mutated, or discarded. The third view is the interpreter view. In that view, we ask what protocol is being used, what bytecode is being executed, and which runtime rules are controlling the result.

Many learners stop after the first view. That is enough to copy code, but it is usually not enough to explain surprising behavior. Deep understanding starts when these three views become one mental movement instead of three unrelated topics.


There is also a fourth habit that helps a lot: failure-based thinking.

For Inheritance, MRO, and Dunder Methods, ask yourself what normally goes wrong when the idea is only half-understood. Does someone confuse identity with equality? Do they expect copying where only rebinding happened? Do they expect a fresh iterator when they really have an exhausted one? Do they trust a default value that is secretly shared? Do they think a class attribute is per-instance state? Once you can predict the common failure modes before they happen, your understanding becomes much more stable.


In [6]:
import dis
import inspect

def probe_runtime(value):
    local_copy = value
    frame = inspect.currentframe()
    print('locals now:', frame.f_locals)
    return local_copy

print(probe_runtime('demo'))
print('\nbytecode for probe_runtime:')
dis.dis(probe_runtime)


locals now: {'value': 'demo', 'local_copy': 'demo', 'frame': <frame at 0x00000119213C7D00, file 'C:\\Users\\VIHAAN\\AppData\\Local\\Temp\\ipykernel_2624\\939914315.py', line 7, code probe_runtime>}
demo

bytecode for probe_runtime:
  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (value)
              4 STORE_FAST               1 (local_copy)

  6           6 LOAD_GLOBAL              0 (inspect)
             18 LOAD_METHOD              1 (currentframe)
             40 PRECALL                  0
             44 CALL                     0
             54 STORE_FAST               2 (frame)

  7          56 LOAD_GLOBAL              5 (NULL + print)
             68 LOAD_CONST               1 ('locals now:')
             70 LOAD_FAST                2 (frame)
             72 LOAD_ATTR                3 (f_locals)
             82 PRECALL                  2
             86 CALL                     2
             96 POP_TOP

  8          98 LOAD_FAST         

The deeper object-model gain here is to inspect instance namespaces, class namespaces, method resolution order, and descriptor objects directly. That shifts the topic from ?class syntax? to ?runtime lookup process?, which is where the real explanatory power lives.


In [7]:
class Base:
    kind = 'base'
    def hello(self):
        return 'hello'

class Child(Base):
    pass

obj = Child()
print('instance dict:', obj.__dict__)
print('class dict sample:', list(Child.__dict__.keys())[:8])
print('mro:', Child.__mro__)
print('bound method:', obj.hello)


instance dict: {}
class dict sample: ['__module__', '__doc__']
mro: (<class '__main__.Child'>, <class '__main__.Base'>, <class 'object'>)
bound method: <bound method Base.hello of <__main__.Child object at 0x0000011921594FD0>>


A deeper reading of inheritance is really a deeper reading of lookup order. Python does not guess what a subclass should do. It follows the method resolution order. Dunder methods also fit this runtime-centered view because they are lookup hooks that let ordinary syntax map onto object behavior. This makes the whole chapter less about memorizing magic names and more about understanding how the language asks objects to participate.


## Final Takeaway

The deepest way to revise **Inheritance, MRO, and Dunder Methods** is to stop treating it as a collection of syntax facts and instead treat it as a runtime story. Ask what objects are involved, what names or attributes point to those objects, what frame is active, and what protocol or bytecode path Python is following. If you can explain this topic at those four levels, then you understand much more than the visible syntax.

In interview terms, the strongest answer is usually not the shortest definition. It is a calm explanation of what Python is doing under the surface and why the behavior follows from that model. For revision, come back to this notebook and ask yourself: could I explain this entire chapter with one small code example, one memory-level explanation, and one interpreter-level explanation? If yes, the topic is becoming yours.
